# 1. Preparação dos dados
Este notebook reúne os artigos do WIE e do WEI. Nenhum artigo será excluído e os textos não serão alterados.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Localiza a pasta raiz do projeto, que deve conter o arquivo README.md e a pasta dados
inicio = Path.cwd().resolve()
raiz = next((p for p in (inicio, *inicio.parents) if (p / 'README.md').exists() and (p / 'dados').exists()), None)
if raiz is None:
    raise FileNotFoundError('Não foi possível localizar a pasta do projeto.')
sys.path.insert(0, str(raiz))
from apoio.funcoes import normalizar_nome_coluna

# Define as pastas de dados brutos e processados
pasta_brutos = raiz / 'dados' / '0_brutos'
pasta_processados = raiz / 'dados' / '1_processados'

### 1.1 Leitura e consolidação das planilhas

In [ ]:
# Verifica se os arquivos de dados brutos existem
arquivos = {
    'WIE': pasta_brutos / 'wie_artigos_consolidados.xlsx',
    'WEI': pasta_brutos / 'wei_artigos_consolidados.xlsx',
}
for arquivo in arquivos.values():
    if not arquivo.exists():
        raise FileNotFoundError(f'Arquivo não encontrado: {arquivo}')

In [ ]:
tabelas = []
resumo_abas = []

# Lê os arquivos Excel e normaliza os nomes das colunas
for evento, arquivo in arquivos.items():
    abas = pd.read_excel(arquivo, sheet_name=None, engine='openpyxl')
    for nome_aba, tabela in abas.items():
        tabela = tabela.copy()
        tabela.columns = [normalizar_nome_coluna(c) for c in tabela.columns]
        tabela.insert(0, 'linha_origem', range(2, len(tabela) + 2))
        tabela.insert(0, 'aba_origem', str(nome_aba))
        tabela.insert(0, 'arquivo_origem', arquivo.name)
        tabela.insert(0, 'evento', evento)
        tabelas.append(tabela)
        resumo_abas.append({'evento': evento, 'aba': nome_aba, 'quantidade': len(tabela)})

# Concatena todas as tabelas em um único DataFrame e adiciona uma coluna de ID
df = pd.concat(tabelas, ignore_index=True, sort=False)
df.insert(0, 'id_artigo', range(1, len(df) + 1))
resumo_abas = pd.DataFrame(resumo_abas)
df.head()

### 1.2 Validação da consolidação

In [ ]:
resumo_eventos = df.groupby('evento').size().rename('quantidade').reset_index()
resumo_eventos.loc[len(resumo_eventos)] = ['TOTAL', len(df)]
resumo_eventos

In [ ]:
resumo_abas

In [ ]:
# Verifica se todas as colunas obrigatórias estão presentes no DataFrame
colunas_obrigatorias = ['ano', 'titulo', 'resumo', 'palavras_chave']
ausentes = [c for c in colunas_obrigatorias if c not in df.columns]
if ausentes:
    raise ValueError(f'Colunas obrigatórias ausentes: {ausentes}')
df[colunas_obrigatorias].isna().sum().rename('valores_ausentes').to_frame()

In [ ]:
# Identifica títulos duplicados, ignorando diferenças de maiúsculas/minúsculas e espaços em branco
titulo_comparacao = df['titulo'].fillna('').astype(str).str.strip().str.casefold()
duplicados = df[titulo_comparacao.ne('') & titulo_comparacao.duplicated(keep=False)]
duplicados[['id_artigo', 'evento', 'ano', 'aba_origem', 'linha_origem', 'titulo']]

### 1.3 Exportação
A saída será a entrada do próximo notebook.

In [ ]:

pasta_processados.mkdir(parents=True, exist_ok=True)
df.to_csv(pasta_processados / '01_artigos_consolidados.csv', index=False, encoding='utf-8-sig')

print(f'Foram exportados {len(df)} registros.')